In [1]:
import torch
import torch.nn as nn
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
# ── 1. Device setup ─────────────────────────────────
device = (
    "cuda"  if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Device: {device}")

Device: cpu


In [3]:
# ── 2. Load crop data (from Week 1) ─────────────────
df = pd.read_csv("../data/crop_recommendation.csv")
le = LabelEncoder()
df["label_enc"] = le.fit_transform(df["label"])

X = df[["N","P","K","temperature","humidity","ph","rainfall"]].values
y = df["label_enc"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

In [4]:
# ── 3. Convert to PyTorch tensors ───────────────────
X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32).to(device)
y_test_t  = torch.tensor(y_test,  dtype=torch.long).to(device)

print(f"Train shape: {X_train_t.shape}")  # [1760, 7]
print(f"Test shape:  {X_test_t.shape}")   # [440, 7]
print(f"Classes:     {len(le.classes_)}")

Train shape: torch.Size([1760, 7])
Test shape:  torch.Size([440, 7])
Classes:     22


In [5]:
# ── 4. Build CropNet ────────────────────────────────
class CropNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(7, 64),
            nn.ReLU(),
            nn.Dropout(0.2),       # prevents overfitting
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 22),   # 22 crops
        )
    def forward(self, x):
        return self.net(x)

model = CropNet().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Parameters: 3,318


In [6]:
# ── 5. Training loop ────────────────────────────────
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(1, 51):
    model.train()
    pred = model(X_train_t)
    loss = criterion(pred, y_train_t)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            acc = (model(X_test_t).argmax(1) == y_test_t).float().mean()
        print(f"Epoch {epoch:3d} | Loss: {loss:.4f} | Acc: {acc:.2%}")

# ── 6. Save model ───────────────────────────────────
torch.save(model.state_dict(), "../models/cropnet_tabular.pth")
print("Model saved!")

Epoch  10 | Loss: 3.0276 | Acc: 7.27%
Epoch  20 | Loss: 2.9348 | Acc: 24.32%
Epoch  30 | Loss: 2.8129 | Acc: 43.18%
Epoch  40 | Loss: 2.6379 | Acc: 51.82%
Epoch  50 | Loss: 2.4133 | Acc: 53.18%
Model saved!


In [7]:
import shutil, os
from sklearn.model_selection import train_test_split

src = "../data/plant_village/PlantVillage/PlantVillage"  # original folder

for cls in os.listdir(src):
    imgs = os.listdir(f"{src}/{cls}")
    train_imgs, val_imgs = train_test_split(imgs, test_size=0.2,
                                            random_state=42)
    for split, lst in [("train", train_imgs), ("val", val_imgs)]:
        dest = f"../data/plant_village/{split}/{cls}"
        os.makedirs(dest, exist_ok=True)
        for img in lst:
            shutil.copy(f"{src}/{cls}/{img}", dest)

    full_path = os.path.join(src, cls)
    if os.path.isdir(full_path):
        imgs = os.listdir(full_path)
        print(f"Found category: {cls} with {len(imgs)} images")

print("Split complete!")

Found category: Pepper__bell___Bacterial_spot with 997 images
Found category: Pepper__bell___healthy with 1478 images
Found category: Potato___Early_blight with 1000 images
Found category: Potato___healthy with 152 images
Found category: Potato___Late_blight with 1000 images
Found category: Tomato_Bacterial_spot with 2127 images
Found category: Tomato_Early_blight with 1000 images
Found category: Tomato_healthy with 1591 images
Found category: Tomato_Late_blight with 1909 images
Found category: Tomato_Leaf_Mold with 952 images
Found category: Tomato_Septoria_leaf_spot with 1771 images
Found category: Tomato_Spider_mites_Two_spotted_spider_mite with 1676 images
Found category: Tomato__Target_Spot with 1404 images
Found category: Tomato__Tomato_mosaic_virus with 373 images
Found category: Tomato__Tomato_YellowLeaf__Curl_Virus with 3209 images
Split complete!


In [8]:
import os
total = sum(len(files) for _,_,files in os.walk("../data/plant_village/train"))
print(f"Total images: {total}")  

Total images: 16505
